# Bakemonogatari EP01 — Style Discovery Exploration

This notebook explores visual style patterns in Bakemonogatari EP01
using the VSDE (Visual Style Differential Engine).

In [ ]:
from vsde.modules import VideoLoader, ShotSegmenter, KeyframeExtractor, Embedder
from vsde.config import config

config.log_settings()

In [ ]:
# Step 1: Load video metadata
video_path = "../data/raw/bakemonogatari_ep01.mp4"
loader = VideoLoader(video_path)
metadata = loader.load()
print(f"FPS: {metadata.fps}, Duration: {metadata.duration_sec:.1f}s, Resolution: {metadata.resolution}")

In [ ]:
# Step 2: Segment into shots
segmenter = ShotSegmenter(metadata)
shots = segmenter.segment()
print(f"Detected {len(shots)} shots")

In [ ]:
# Step 3: Extract keyframes

extractor = KeyframeExtractor(metadata, shots, output_dir=config.FRAMES_DIR / "target")
keyframe_paths = extractor.extract()
print(f"Extracted {len(keyframe_paths)} keyframes")

In [ ]:
# Step 4: Compute embeddings

embedder = Embedder(model_type="concat")
embeddings = embedder.embed_frames(keyframe_paths)
print(f"Embeddings shape: {embeddings.shape}")

In [ ]:
# Step 5: Build or load the baseline reference vector

from vsde.modules import BaselineBuilder, DifferentialEngine
from vsde.config import BASELINE_CACHE_DIR

baseline_builder = BaselineBuilder(cache_dir=BASELINE_CACHE_DIR)

# Option A: Build from cached baseline embeddings
#   (run this once per reference work, result is cached)
# baseline_builder.load_embeddings_from_cache(video_id="hibike_ep01")
# baseline, path = baseline_builder.build_and_save(name="hibike_baseline", strategy="mean")

# Option B: Load a saved baseline
baseline_meta = BaselineBuilder.load(name="hibike_baseline", cache_dir=BASELINE_CACHE_DIR)
print(f"Loaded baseline '{baseline_meta.name}' ({baseline_meta.shot_count} shots, strategy={baseline_meta.strategy})")

In [ ]:
# Step 6: Compare target shots against the baseline

engine = DifferentialEngine(target_embeddings=embeddings)
engine.set_baseline_vector(baseline_meta.baseline_vector, name=baseline_meta.name)
engine.set_shots(target_shots=shots, baseline_shots=[])

diff_results = engine.compare(mode="vs_baseline")
print(f"vs_baseline: {len(diff_results)} shots analyzed")
top = diff_results[0]
print(f"Top deviation: shot={top.shot_id}, magnitude={top.diff_magnitude:.4f}, baseline={top.baseline_name}")